# The golden routing dataset

One row per labelled `(dataset, query_id)`: the three per-route objective
scores, the derived route, and the outcome shape. A label comes from running
`dense_only` / `pure_rrf` / `sparse_only` against the lane's indexed corpus
and scoring each ranking with `0.7·HitRate@1 + 0.3·NDCG@10` — never from
asking a model which route looks right. The build pipeline lives in
`notebooks/route_labels.ipynb`; this notebook only reads the artifact.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd

from composition import CellFill
from hybrid_search_rrf_dataset.labels import RouteLabels
from hybrid_search_rrf_dataset.objective import RouterObjective

selection = CellFill().build()
labels = RouteLabels(selection, objective=RouterObjective(min_relevance=1))
data = labels.load()
print(f"{len(data):,} labelled rows | {data['dataset'].nunique()} lanes | "
      f"{data['query_id'].nunique():,} distinct queries")
print(f"objective: {labels.objective.name}")

## 1 — What's in it

Shape decides what a row can teach. `routes_differ` carries the
dense-vs-sparse signal, `all_tied` says any route works, `all_zero` found
nothing relevant and stores no label.

In [ ]:
shapes = data["shape"].value_counts().rename_axis("shape").to_frame("rows")
shapes["share"] = (shapes["rows"] / len(data) * 100).round(1)
shapes

In [ ]:
differ = data[data["shape"] == "routes_differ"]
routes = differ["route"].value_counts().rename_axis("route").to_frame("wins")
routes["share of trainable"] = (routes["wins"] / len(differ) * 100).round(1)
routes

The corpora are why routing has something to learn: lanes differ in average
IDF and out-of-vocabulary share, and the best blind route moves with them.

In [ ]:
stats = pd.read_parquet("data/route_labels/lane_corpus_stats.parquet")
print(stats.groupby("target")["avg_idf"].agg(["count", "mean"]).round(3).to_string())
pd.concat([stats.nlargest(3, "avg_idf"), stats.nsmallest(3, "avg_idf")])[
    ["dataset", "avg_idf", "oov_share", "target"]
].round(3)

## 2 — One winner per query: the decisive core

The argmax form keeps a row for training only when one route clearly won.
Decisive means the winner put a relevant document at rank 1 and the
runner-up missed — under this objective exactly margin ≥ 0.4, derived from
the weights, not hand-picked. The bar is deliberately strict, and it keeps
about one row in nine. Twice the pre-repair count (the IDF and
selection-join fixes), but the other eight rows stay invisible to training.

In [ ]:
from hybrid_search_rrf_dataset.router import decisive_rows

before = pd.read_parquet(labels.labels_path.parent / "labels.backup-pre-idf-full.parquet")
now, then = decisive_rows(data), decisive_rows(before)
print(f"decisive now:        {len(now):,} of {len(data):,} rows")
print(f"decisive before fix: {len(then):,} of {len(before):,} rows")
now["winner"].value_counts().rename_axis("winner").to_frame("decisive wins")

## 3 — Three yes/no labels: ties become signal

SPEC d60 changes the label form, not the scores. Each route gets
`ok = score ≥ oracle − 0.3`, where 0.3 is the objective's own `ndcg_weight`
— the widest gap two routes can show while still sharing the same top-1
outcome. A tied row labels `[1, 1, 1]` and trains all three classifiers;
`all_zero` rows stay null. At tolerance 0 the view reproduces the stored
`route` column exactly, so nothing is overwritten — the same scores read a
second way.

In [16]:
view = labels.acceptability().frame()
answerable = view[view["serve"].notna()]
print(f"decisive rows:      {len(now):,}")
print(f"acceptability rows: {len(answerable):,}  "
      f"({len(answerable) / len(now):.1f}x, zero new retrieval)")

pd.DataFrame({
    route: answerable[f"ok_{route}"].astype(bool).value_counts()
    for route in ("dense_only", "pure_rrf", "sparse_only")
}).T.rename(columns={True: "acceptable", False: "not acceptable"})

decisive rows:      5,100
acceptability rows: 37,961  (7.4x, zero new retrieval)


,acceptable,not acceptable
dense_only,33366,4595
pure_rrf,33077,4884
sparse_only,29482,8479


`serve` — the cheapest acceptable route — is the evaluation target, not a
training label. On tied rows it points at sparse, which is where the class
that argmax starved gets its training diet.

In [17]:
serve = answerable["serve"].value_counts().rename_axis("serve").to_frame("rows")
serve["share"] = (serve["rows"] / len(answerable) * 100).round(1)
serve

,rows,share
serve,,
sparse_only,29482,77.7
dense_only,8259,21.8
pure_rrf,220,0.6


## 4 — Growing the thin archetypes: augmentation

Half the archetype cells cannot fill their draw from natural supply
(`notebooks/selection_audit.ipynb` §3); those cells are generation targets. An
operator edits a real parent query under a declared meaning-preserving
transformation, so the child inherits the parent's human judgments (d43d).
`data/augmentation/pool.parquet` holds the accepted children.

In [18]:
from composition.cells import CELLS_BY_NAME
from composition.compose import join_text
from dataset_registry import DATASETS

pool = pd.read_parquet("data/augmentation/pool.parquet")
for_cells = pool[pool["floor"].isin(CELLS_BY_NAME)]
print(f"{len(pool):,} accepted children | {len(for_cells):,} minted for "
      f"{for_cells['floor'].nunique()} archetype cells, the rest for "
      f"pre-cell band floors")

sample = for_cells.groupby("floor").head(1).head(6).copy()
as_parents = sample.drop(columns=["query_id"]).rename(
    columns={"parent_dataset": "dataset", "generated_from": "query_id"})
sample["parent"] = join_text(as_parents, {d.name: d for d in DATASETS}).values
sample.rename(columns={"floor": "cell"})[["cell", "operator", "parent"]].assign(
    parent=sample["parent"].str.slice(0, 70),
    child=sample["query"].str.slice(0, 70),
)

2,365 accepted children | 376 minted for 22 archetype cells, the rest for pre-cell band floors


,cell,operator,parent,child
1989,legal_citation_canonical,inject,Are eviction cases first heard in circuit cour...,North Carolina eviction proceedings 42 U.S.C. ...
2019,datetime_token_present,inject,[Image],events on 2017-03-31 23:26:44
2031,single_token_char_blob,stat_rewrite,Why do people drink Cuervo when it tastes the ...,Cuervo_identical_taste_ingestion_regurgitation...
2039,business_temporal_reference,inject,2021 films about action and comedy.,action and comedy films released during Q4 2021
2044,symbol_pile_no_grammar,inject,alternative medicine,cPGES lipoxinA4 alternative medicine
2074,bibliographic_catalog_identifier,inject,Problem: Brook Hills High School currently enr...,Brook Hills High School 3000-1125 female stude...


### One round, priced before it is paid for

`campaign.plan()` shows the whole pass — one row per hungry floor with the
operator it would use and the row count it targets — without an LLM call.
The commented line underneath runs one real round for a single floor and
appends its accepted children to the pool.

In [19]:
from augmentation.campaign import AugmentationCampaign
from augmentation.config import AugmentationPaths
from augmentation.loop import AugmentationLoop
from augmentation.parents import ParentPool
from dataset_registry import DATASETS

paths = AugmentationPaths()
catalog = pd.read_parquet(paths.catalog).astype({"query_id": str})
parents = ParentPool(catalog, selection.astype({"query_id": str}),
                     {d.name: d for d in DATASETS})
loop = AugmentationLoop(selection, sheet_path=paths.cell_order_sheet, parents=parents)

plan = AugmentationCampaign(loop).plan()
plan.head(10)

parents: dropped 3 row(s) with no query text
parents: dropped 8 row(s) with no query text
parents: dropped 2 row(s) with no query text
parents: dropped 1 row(s) with no query text
parents: dropped 6 row(s) with no query text
parents: dropped 5 row(s) with no query text
parents: dropped 2 row(s) with no query text
campaign plan — 22 hungry floors, 284 target rows (>= 284 LLM calls):
                           floor  missing                operator              gate             action  target_rows
        legal_citation_canonical    400.0    inject, stat_rewrite    coherence_gate skip: pilot staged            0
          datetime_token_present    399.0    inject, stat_rewrite    coherence_gate            produce           13
          single_token_char_blob    398.0            stat_rewrite declaration_audit            produce           22
     business_temporal_reference    398.0            stat_rewrite declaration_audit            produce           25
          symbol_pile_no_grammar   

,floor,missing,operator,gate,action,target_rows
0,legal_citation_canonical,400.0,"inject, stat_rewrite",coherence_gate,skip: pilot staged,0
1,datetime_token_present,399.0,"inject, stat_rewrite",coherence_gate,produce,13
2,single_token_char_blob,398.0,stat_rewrite,declaration_audit,produce,22
3,business_temporal_reference,398.0,stat_rewrite,declaration_audit,produce,25
4,symbol_pile_no_grammar,398.0,"inject, stat_rewrite",coherence_gate,skip: pilot staged,0
5,bibliographic_catalog_identifier,398.0,"inject, stat_rewrite",coherence_gate,skip: pilot staged,0
6,bio_clinical_identifier,398.0,"inject, stat_rewrite",coherence_gate,skip: pilot staged,0
7,travel_transport_code,397.0,"inject, stat_rewrite",coherence_gate,produce,25
8,standards_compliance_lookup,394.0,"inject, stat_rewrite",coherence_gate,produce,9
9,boolean_operator_query,393.0,operator_syntax_rewrite,declaration_audit,produce,28


In [ ]:
# one real round for the first plannable floor — spends LLM tokens:
floor = plan[plan["action"] == "produce"].iloc[0]["floor"]
loop.run(floor, n=1)

parents: dropped 3 row(s) with no query text
NOTE: 'inject' is gated by coherence_gate (d42h) — rows are produced as feature-stock (no floor credit; skipped by the mini-fill) until the gate's pilot passes.
NOTE: 'stat_rewrite' is gated by declaration_audit (d42h) — rows are produced as feature-stock (no floor credit; skipped by the mini-fill) until the gate's pilot passes.


augment:datetime_token_present:   0%|          | 0/1 [00:01<?, ?row/s, attempted=1, dropped=1]

- 2046371: dropped — failed [length_words >= 3.0 and <= 9.999999999 (measured 12.0)]
    before: 'functions of three regions of sm intestine'
    tried:  'functions of three sm intestine regions 2014-09-17 16:59:58'


augment:datetime_token_present:   0%|          | 0/1 [00:12<?, ?row/s, attempted=2, dropped=2]

- 372: dropped — rounds_exhausted — rounds called: ['generate_surface', 'list_features', 'verify', 'verify', 'generate_surface', 'verify']
    before: 'You are given two circles. Find the area of their intersection.'
    tried:  (nothing — never got that far)


...

- 2599: dropped — rounds_exhausted — rounds called: ['list_features', 'generate_surface', 'generate_surface', 'verify', 'generate_surface', 'verify']
    before: 'Let $f(x)$ be the sum of digits of a decimal number $x$.\n\nFind the smallest non-'
    tried:  (nothing — never got that far)


augment:datetime_token_present:   0%|          | 0/1 [00:38<?, ?row/s, attempted=4, dropped=4]

- 539: dropped — rounds_exhausted — rounds called: ['list_features', 'generate_surface', 'verify', 'verify', 'verify', 'verify']
    before: 'Sixties era movie – an Old Lady’s cat inherits her fortune .\n When an old lady d'
    tried:  (nothing — never got that far)


augment:datetime_token_present:   0%|          | 0/1 [00:49<?, ?row/s, attempted=5, dropped=5]

- 1217: dropped — rounds_exhausted — rounds called: ['list_features', 'generate_surface', 'verify', 'verify', 'verify', 'verify']
    before: 'You are given two arrays of integers a and b. For each element of the second arr'
    tried:  (nothing — never got that far)
